In [1]:
from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS
from omegaconf import OmegaConf
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, balanced_accuracy_score, roc_auc_score

dataset_path = "/opt/gpudata/ecg/heedb"
arms = ["2D-partial", "2D-global", "1D-global"]
run_path = Path("/opt/gpu_working/steven/protoecgnet-heedb")
config_path = Path("/opt/gpu_working/steven/ProtoSSL/large-user-study/configs")

In [2]:
def stack_branched_arms():
    arm_labels = [l for arm in arms for l in OmegaConf.load(config_path / f"{arm}.yaml").data.init_args.label_subset]
    idxs = np.asarray([arm_labels.index(l) for l in HEEDB_TARGETS]) # indices to normalize to fusion order

    ret = dict()
    for split in ["val", "test"] :
        arm_probs = {arm: np.load(run_path / arm / f"train-classifier/latest/{split}_probs.npy") for arm in arms}
        stacked_probs = np.concat([arm_probs[arm] for arm in arms], axis=1)
        ret[split] = stacked_probs[:, idxs]
    return ret

In [3]:
ds_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="original_physician",
    heedb_split_type="by-label",
)

ds_phys_val = HeedbECGDataset(
    dataset_path=dataset_path,
    split="val",
    sampling_rate=100,
    label_src="original_physician",
    heedb_split_type="by-label",
)

ds_muse = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src="original_muse",
    heedb_split_type="by-label",
)

================get_heedb_metadata=================
using by-label splits
reading HEEDB metadata from on-disk cache: /home/songs1/.cache/protossl_cache/abac20ad.csv
=================make_heedb_labels=================
reading HEEDB labels from source: /opt/gpudata/ecg/heedb


Converting code string to labels: 100%|██████████| 453512/453512 [00:01<00:00, 287988.11it/s]


saved HEEDB labels to on-disk cache: /home/songs1/.cache/protossl_cache/51ff3555.npy
Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================
reading HEEDB labels from source: /opt/gpudata/ecg/heedb


Converting code string to labels: 100%|██████████| 453512/453512 [00:02<00:00, 170936.84it/s]


saved HEEDB labels to on-disk cache: /home/songs1/.cache/protossl_cache/c6af0872.npy
Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
=================make_heedb_labels=================
reading HEEDB labels from source: /opt/gpudata/ecg/heedb


Converting code string to labels: 100%|██████████| 453512/453512 [00:01<00:00, 298284.00it/s]


saved HEEDB labels to on-disk cache: /home/songs1/.cache/protossl_cache/0a73c965.npy
Of 453512 ECGs and 11607259 annotations, 453512 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly


In [4]:
y_true = ds_phys.labels.numpy()
y_true_val = ds_phys_val.labels.numpy()
y_pred_muse = ds_muse.labels.numpy()

In [5]:
def compute_threshold(_y_true, _y_prob):
    # NOTE: should be computed on either train/val set, NOT test
    fpr, tpr, thresholds = roc_curve(_y_true, _y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold

In [6]:
temp = stack_branched_arms()
y_prob_branched, y_prob_branched_val = temp["test"], temp["val"]
y_prob_fusion = np.load(run_path / "fusion/train-fusion-classifier/latest/test_probs.npy")
y_prob_fusion_val = np.load(run_path / "fusion/train-fusion-classifier/latest/val_probs.npy")

y_pred_branched = np.zeros_like(y_pred_muse)
y_pred_fusion = np.zeros_like(y_pred_muse)
accuracy_results = []
roc_results = []
for i, label in enumerate(HEEDB_TARGETS):
    branched_threshold = compute_threshold(y_true_val[:, i], y_prob_branched_val[:, i])
    y_pred_branched[:, i] = (y_prob_branched[:, i] > branched_threshold).astype(int)
    fusion_threshold = compute_threshold(y_true_val[:, i], y_prob_fusion_val[:, i])
    y_pred_fusion[:, i] = (y_prob_fusion[:, i] > fusion_threshold).astype(int)
    label_acc_results = {
        "label": label,
        "muse: BalAcc": balanced_accuracy_score(y_true[:, i], y_pred_muse[:, i]),
        "branched: BalAcc": balanced_accuracy_score(y_true[:, i], y_pred_branched[:, i]),
        "fusion: BalAcc": balanced_accuracy_score(y_true[:, i], y_pred_fusion[:, i]),
    }
    label_roc_results = {
        "label": label,
        # NOTE: can't compute muse roc given no probs
        "branched: AUROC": roc_auc_score(y_true[:, i], y_prob_branched[:, i]),
        "fusion: AUROC": roc_auc_score(y_true[:, i], y_prob_fusion[:, i]),
    }
    accuracy_results.append(label_acc_results)
    roc_results.append(label_roc_results)

In [10]:
roc_df = pd.DataFrame(roc_results)
idx = len(roc_df)
roc_df.loc[idx, "label"] = "Macro Average"
for c in [c for c in roc_df.columns if c != "label"]:
    roc_df.loc[idx, c] = roc_df[c].mean()

with pd.option_context("display.precision", 3):
    display(roc_df)

,label,branched: AUROC,fusion: AUROC
0,ANTERIOR INFARCT,0.953,0.952
1,ATRIAL FIBRILLATION,0.958,0.958
2,ATRIAL FLUTTER,0.952,0.951
3,ATRIAL-PACED RHYTHM,0.987,0.987
4,INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,0.962,0.962
5,INFERIOR INFARCT,0.976,0.975
6,LATERAL INFARCT,0.957,0.956
7,LEFT BUNDLE BRANCH BLOCK,0.987,0.986
8,NORMAL SINUS RHYTHM,0.924,0.924
9,PREMATURE ATRIAL COMPLEXES,0.901,0.908


In [11]:
acc_df = pd.DataFrame(accuracy_results)
idx = len(acc_df)
acc_df.loc[idx, "label"] = "Macro Average"
for c in [c for c in acc_df.columns if c != "label"]:
    acc_df.loc[idx, c] = acc_df[c].mean()

with pd.option_context("display.precision", 3):
    display(acc_df)

,label,muse: BalAcc,branched: BalAcc,fusion: BalAcc
0,ANTERIOR INFARCT,0.949,0.891,0.891
1,ATRIAL FIBRILLATION,0.921,0.904,0.904
2,ATRIAL FLUTTER,0.858,0.889,0.888
3,ATRIAL-PACED RHYTHM,0.873,0.945,0.946
4,INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,0.910,0.906,0.906
5,INFERIOR INFARCT,0.972,0.919,0.918
6,LATERAL INFARCT,0.973,0.908,0.909
7,LEFT BUNDLE BRANCH BLOCK,0.940,0.958,0.957
8,NORMAL SINUS RHYTHM,0.875,0.857,0.857
9,PREMATURE ATRIAL COMPLEXES,0.910,0.835,0.846
